In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
# Load the saved Tuned Model

loaded_model = joblib.load("best_housing_model.pkl")

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [3]:
# Load Cleaned Housing Dataset

df = pd.read_csv("cleaned_housing_data.csv", keep_default_na=False)

# Convert MSSubClass to string
df["MSSubClass"] = df["MSSubClass"].astype(str)

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

Dataset Loaded Successfully!
Dataset Shape: (1460, 81)


In [4]:
from sklearn.model_selection import train_test_split

# Separate Features and Target
X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

# Same Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Testing Data Shape:", X_test.shape)
print("Actual Target Shape:", y_test.shape)

Testing Data Shape: (292, 80)
Actual Target Shape: (292,)


In [5]:
# Select one house from testing data

new_house = X_test.iloc[[0]]

# Get its actual price
actual_price = y_test.iloc[0]

print("House Features:")
print(new_house)

print("\nActual House Price:", actual_price)

House Features:
      Id MSSubClass MSZoning  LotFrontage  LotArea Street    Alley LotShape  \
892  893         20       RL         70.0     8414   Pave  NoAlley      Reg   

    LandContour Utilities  ... ScreenPorch PoolArea  PoolQC  Fence  \
892         Lvl    AllPub  ...           0        0  NoPool  MnPrv   

    MiscFeature MiscVal MoSold  YrSold  SaleType  SaleCondition  
892   NoFeature       0      2    2006        WD         Normal  

[1 rows x 80 columns]

Actual House Price: 154500


In [6]:
# Predict House Price

predicted_price = loaded_model.predict(new_house)

print("Predicted House Price:", predicted_price[0])

Predicted House Price: 140472.66333333333


In [7]:
# Calculate Prediction Error

prediction_error = actual_price - predicted_price[0]

absolute_error = abs(prediction_error)

percentage_error = (
    absolute_error / actual_price
) * 100

print("Actual Price:", actual_price)
print("Predicted Price:", round(predicted_price[0], 2))
print("Absolute Error:", round(absolute_error, 2))
print("Percentage Error:", round(percentage_error, 2), "%")

Actual Price: 154500
Predicted Price: 140472.66
Absolute Error: 14027.34
Percentage Error: 9.08 %


In [8]:
# Create a Prediction Summary

prediction_summary = pd.DataFrame({
    "House_ID": [new_house["Id"].iloc[0]],
    "Actual_Price": [actual_price],
    "Predicted_Price": [round(predicted_price[0], 2)],
    "Absolute_Error": [round(absolute_error, 2)],
    "Percentage_Error": [round(percentage_error, 2)]
})

print(prediction_summary)

   House_ID  Actual_Price  Predicted_Price  Absolute_Error  Percentage_Error
0       893        154500        140472.66        14027.34              9.08


In [ ]:
prediction_summary.to_csv(
    "prediction_testing_result.csv",
    index=False
)

print("Prediction Result Saved Successfully!")

In [9]:
# Select 20 unseen houses from test data

multiple_houses = X_test.head(20).copy()
actual_prices = y_test.head(20)

print("Selected Houses:", multiple_houses.shape)

Selected Houses: (20, 80)


In [10]:
# Predict prices for 20 houses

multiple_predictions = loaded_model.predict(multiple_houses)

print("Total Predictions:", len(multiple_predictions))
print("First 5 Predictions:", multiple_predictions[:5])

Total Predictions: 20
First 5 Predictions: [140472.66333333 323087.84       114479.33333333 154088.35
 325661.60666667]


In [11]:
# Create Prediction Results Table

multiple_results = pd.DataFrame({
    "House_ID": multiple_houses["Id"].values,
    "Actual_Price": actual_prices.values,
    "Predicted_Price": np.round(multiple_predictions, 2)
})

print(multiple_results)

    House_ID  Actual_Price  Predicted_Price
0        893        154500        140472.66
1       1106        325000        323087.84
2        414        115000        114479.33
3        523        159000        154088.35
4       1037        315500        325661.61
5        615         75500         85403.00
6        219        311500        207840.12
7       1161        146000        151670.26
8        650         84500         86100.38
9        888        135500        129328.99
10       577        145000        156298.42
11      1253        130000        120647.90
12      1062         81000        108272.83
13       568        214000        207742.35
14      1109        181000        177030.38
15      1114        134500        129943.92
16       169        183500        195867.20
17      1103        135000        136630.12
18      1121        118400        113721.17
19        68        226000        204279.12


In [12]:
# Calculate prediction errors

multiple_results["Absolute_Error"] = (
    multiple_results["Actual_Price"] -
    multiple_results["Predicted_Price"]
).abs()

multiple_results["Percentage_Error"] = (
    multiple_results["Absolute_Error"] /
    multiple_results["Actual_Price"] * 100
).round(2)

print(multiple_results)

    House_ID  Actual_Price  Predicted_Price  Absolute_Error  Percentage_Error
0        893        154500        140472.66        14027.34              9.08
1       1106        325000        323087.84         1912.16              0.59
2        414        115000        114479.33          520.67              0.45
3        523        159000        154088.35         4911.65              3.09
4       1037        315500        325661.61        10161.61              3.22
5        615         75500         85403.00         9903.00             13.12
6        219        311500        207840.12       103659.88             33.28
7       1161        146000        151670.26         5670.26              3.88
8        650         84500         86100.38         1600.38              1.89
9        888        135500        129328.99         6171.01              4.55
10       577        145000        156298.42        11298.42              7.79
11      1253        130000        120647.90         9352.10     

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

actual = multiple_results["Actual_Price"]
predicted = multiple_results["Predicted_Price"]

mae_20 = mean_absolute_error(actual, predicted)
rmse_20 = np.sqrt(mean_squared_error(actual, predicted))
r2_20 = r2_score(actual, predicted)

print("20 Houses Evaluation")
print("MAE:", round(mae_20, 2))
print("RMSE:", round(rmse_20, 2))
print("R2 Score:", round(r2_20, 4))

20 Houses Evaluation
MAE: 13082.08
RMSE: 25424.6
R2 Score: 0.8794


In [14]:
multiple_results.to_csv(
    "multiple_prediction_testing_results.csv",
    index=False
)

print("20 Houses Prediction Results Saved Successfully!")

20 Houses Prediction Results Saved Successfully!


In [15]:
# Predict prices for all 292 test houses

all_test_predictions = loaded_model.predict(X_test)

print("Total Test Houses:", len(all_test_predictions))
print("Actual Test Houses:", len(y_test))

Total Test Houses: 292
Actual Test Houses: 292


In [16]:
# Evaluate saved model on all 292 test houses

final_mae = mean_absolute_error(y_test, all_test_predictions)

final_rmse = np.sqrt(mean_squared_error(y_test, all_test_predictions))

final_r2 = r2_score(y_test, all_test_predictions)

print("Final Saved Model Evaluation")
print("MAE:", round(final_mae, 2))
print("RMSE:", round(final_rmse, 2))
print("R2 Score:", round(final_r2, 4))

Final Saved Model Evaluation
MAE: 17447.31
RMSE: 28909.14
R2 Score: 0.891


In [17]:
# Check Python Environment

import sys

print("Python Version:", sys.version)
print("Python Environment:", sys.executable)

Python Version: 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Python Environment: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3


In [18]:
# Install Dashboard Libraries

%pip install streamlit plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 18.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.8/39.8 MB 19.3 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 23.7 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 11.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [streamlit]12 [streamlit]

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import streamlit as st
import plotly

print("Streamlit Version:", st.__version__)
print("Plotly Version:", plotly.__version__)
print("Dashboard Libraries Ready!")

Streamlit Version: 1.67.0
Plotly Version: 7.1.0
Dashboard Libraries Ready!


In [20]:
import os

print("Current Project Location:", os.getcwd())

Current Project Location: /Users/drashtidarji/Documents/my projects/Housing Prices Prediction/Housing_Price_Prediction_Project
